In [6]:
import pandas as pd
import numpy as np
import re
import urllib.parse

# --- CONFIGURACIÓN ---
table_title = "Exoplanet Catalogue in the Solar Neighbourhood (d < 10pc)"
file_csv_limpio = 'Exoplanets_SolarNeighbourhood _Catalogue.csv'

print(f"Leyendo catálogo maestro: {file_csv_limpio}...")
df = pd.read_csv(file_csv_limpio)

# -------------------------------------------------------------------------
# 1. FUNCIÓN DE FORMATO (Estética de cifras significativas)
# -------------------------------------------------------------------------
def format_val_and_error(val, err_min, err_max):
    if pd.isna(val): return "", ""
    
    # --- EL ESCUDO ANTI-STRINGS (Límites <, >, ~) ---
    if isinstance(val, str):
        val_clean = val.strip()
        try:
            # Intentamos convertir a float por si es un número guardado como texto
            val = float(val_clean)
        except ValueError:
            # Si no se puede (tiene un "<" o ">"), lo devolvemos tal cual sin error
            return val_clean, ""
            
    if pd.isna(err_min) and pd.isna(err_max): return f"{val:.4g}", ""
    
    e_min = abs(err_min) if pd.notna(err_min) else np.nan
    e_max = abs(err_max) if pd.notna(err_max) else np.nan
    if pd.isna(e_min): e_min = e_max
    if pd.isna(e_max): e_max = e_min
    
    if pd.isna(e_max) or e_max == 0 or np.isinf(e_max): 
        return f"{val:.4g}", ""

    def get_needed_decimals(x):
        if pd.isna(x) or x == 0 or np.isinf(x): return 2
        try:
            order_of_mag = np.floor(np.log10(x))
            decs = int(2 - 1 - order_of_mag)
            return max(0, min(9, decs))
        except: return 2

    dec_up = get_needed_decimals(e_max)
    dec_lo = get_needed_decimals(e_min)
    decimals = max(dec_up, dec_lo)

    is_symmetric = False
    try:
        mean_err = (e_max + e_min) / 2
        if mean_err == 0: is_symmetric = True; err_use = 0
        elif (abs(e_max - e_min) / mean_err) < 0.05: is_symmetric = True; err_use = mean_err
        else: is_symmetric = False; err_use = max(e_max, e_min)
    except: is_symmetric = True; err_use = e_max

    def fmt(n, dec): return f"{n:.{dec}f}"
    val_str = fmt(val, decimals)

    if is_symmetric:
        err_str = f"&#177; {fmt(err_use, decimals)}"
    else:
        up, lo = fmt(e_max, decimals), fmt(e_min, decimals)
        err_str = (f"<div style='display: inline-block; vertical-align: middle; line-height: 1; font-size: 0.8em;'>"
                   f"<span style='display: block;'>+{up}</span><span style='display: block;'>-{lo}</span></div>")
                   
    return val_str, err_str
def format_reference_smart(ref_html):
    if pd.isna(ref_html) or not str(ref_html).strip():
        return ""
    
    ref_str = str(ref_html).strip()
    ref_lower = ref_str.lower()
    
    # 0. ALERTA TO-DO (Lo primero: si falta algo, salta la alarma roja)
    if "todo" in ref_lower or "to-do" in ref_lower:
        return "<span style='color: #721c24; background-color: #f8d7da; padding: 2px 5px; border-radius: 4px; font-weight: bold; border: 1px solid #f5c6cb;'>TO-DO</span>"

    # Extraemos solo el texto visible por si lo necesitamos para comparar
    text_only = re.sub(r'<[^>]+>', '', ref_lower).strip()

    # 1. IDENTIFICAR CATÁLOGOS BASE Y ASIGNAR SU COLOR
    color_base = None
    if text_only == 'eu' or 'exoplanet.eu' in ref_lower:
        color_base = "#e83e8c" # Rosa
    elif 'nasa' in ref_lower or 'ipac' in ref_lower:
        color_base = "#fd7e14" # Naranja
    elif 'cif' in text_only or 'cif25' in text_only:
        color_base = "#007bff" # Azul
    elif 'gaia' in ref_lower or 'dr2' in ref_lower or 'dr3' in ref_lower or 'dr4' in ref_lower or 'vizier' in ref_lower:
        color_base = "#28a745" # Verde

    # 2. APLICAR EL COLOR A LOS CATÁLOGOS BASE (¡Respetando el enlace!)
    if color_base:
        if "<a " in ref_str:
            # Si tiene enlace, le quitamos estilos viejos y le inyectamos el color base
            ref_str = re.sub(r'style="[^"]*"', '', ref_str)
            return ref_str.replace("<a ", f"<a style='text-decoration: none; font-weight: bold; color: {color_base};' ")
        else:
            # Si no tiene enlace (texto plano), lo envolvemos en un span
            return f"<span style='color: {color_base}; font-weight: bold;'>{ref_str}</span>"
        
    # 3. TUS PARCHES MANUALES: Escala térmica (Azules -> Púrpuras/Índigos)
    if "<a " in ref_str:
        match = re.search(r'\b(199\d|20[0-2]\d)\b', ref_str)
        color_year = "#6f42c1" # Púrpura medio por defecto
        
        if match:
            year = int(match.group(1))
            if year < 2010:          color_year = "#6c757d" # Gris (Papers antiguos)
            elif 2010 <= year <= 2015: color_year = "#5c6bc0" # Azul apagado
            elif 2016 <= year <= 2020: color_year = "#3949ab" # Azul intenso
            elif 2021 <= year <= 2023: color_year = "#8e24aa" # Púrpura vibrante
            else:                      color_year = "#4a148c" # Índigo / Morado oscuro (2024-2025)

        # Inyectamos el color del año en tu enlace
        ref_str = re.sub(r'style="[^"]*"', '', ref_str)
        return ref_str.replace("<a ", f"<a style='text-decoration: none; font-weight: bold; color: {color_year};' ")
        
    # 4. Fallback: Cualquier texto plano suelto que no encaje en nada
    return f"<span style='color: #6c757d; font-weight: bold;'>{ref_str}</span>"

# -------------------------------------------------------------------------
# 2. ENSAMBLAJE DE DATOS
# -------------------------------------------------------------------------
params = {
    'd': {'unit': 'pc', 'val': 'star_distance', 'min': 'star_distance_error_min', 'max': 'star_distance_error_max'},
    'P_orb': {'unit': 'd', 'val': 'orbital_period', 'min': 'orbital_period_error_min', 'max': 'orbital_period_error_max'},
    'a_p': {'unit': 'au', 'val': 'semi_major_axis', 'min': 'semi_major_axis_error_min', 'max': 'semi_major_axis_error_max'},
    'e_p': {'unit': '', 'val': 'eccentricity', 'min': 'eccentricity_error_min', 'max': 'eccentricity_error_max'},
    'i_p': {'unit': 'deg', 'val': 'inclination', 'min': 'inclination_error_min', 'max': 'inclination_error_max'},
    'T_0,p': {'unit': 'BJD', 'val': 'T0_val', 'min': 'T0_min', 'max': 'T0_max'},
    '&#969;_p': {'unit': 'deg', 'val': 'omega', 'min': 'omega_error_min', 'max': 'omega_error_max'},
    'R_p': {'unit': 'R<sub>J</sub>', 'val': 'radius', 'min': 'radius_error_min', 'max': 'radius_error_max'},
    'M_p': {'unit': 'M<sub>J</sub>', 'val': 'mass', 'min': 'mass_error_min', 'max': 'mass_error_max'},
    'R_star': {'unit': 'R<sub>&#9737;</sub>', 'val': 'star_radius', 'min': 'star_radius_error_min', 'max': 'star_radius_error_max'},
    'M_star': {'unit': 'M<sub>&#9737;</sub>', 'val': 'star_mass', 'min': 'star_mass_error_min', 'max': 'star_mass_error_max'},
    '&#961;_star': {'unit': 'g cm<sup>-3</sup>', 'val': 'star_density', 'min': 'star_density_error_min', 'max': 'star_density_error_max'},
    'T_eff': {'unit': 'K', 'val': 'star_teff', 'min': 'star_teff_error_min', 'max': 'star_teff_error_max'}
}

internal_data = {} 

# Procesamos ID Planeta con hipervínculo automático a SIMBAD
nombres_con_enlace = []
for nombre in df['name']:
    nombre_str = str(nombre).strip()
    
    # Codificamos el nombre para la URL (convierte espacios y caracteres raros)
    nombre_url = urllib.parse.quote(nombre_str)
    
    # Montamos la URL exacta de búsqueda de SIMBAD
    url_simbad = f"https://simbad.cds.unistra.fr/simbad/sim-basic?Ident={nombre_url}"
    
    # Creamos el enlace HTML (lo ponemos en azul oscuro y negrita para que parezca clicable)
    enlace_html = f"<a href='{url_simbad}' target='_blank' style='color: #0056b3; font-weight: bold; text-decoration: none;'>{nombre_str}</a>"
    
    nombres_con_enlace.append(enlace_html)

internal_data['id_planeta'] = nombres_con_enlace

# Procesamos Discovery (referencia del descubrimiento)
discovery_refs = []
for idx, row in df.iterrows():
    raw_ref = row.get('discovery_ref', '')
    raw_ref = str(raw_ref) if pd.notna(raw_ref) else ''
    discovery_refs.append(format_reference_smart(raw_ref))
internal_data['discovery_ref'] = discovery_refs

internal_data['metodo'] = df['Detection_Method_Full'].tolist()

# Procesamos Enlaces a Bases de Datos (EU y NASA)
eu_links = []
nasa_links = []

for nombre in df['name']:
    nombre_str = str(nombre).strip()
    
    # URL de Exoplanet.eu: Suelen sustituir los espacios por barras bajas (_)
    nombre_eu = nombre_str.replace(" ", "_")
    url_eu = f"http://exoplanet.eu/catalog/{nombre_eu}/"
    # Lo pintamos con su color corporativo naranja
    eu_html = f"<a href='{url_eu}' target='_blank' style='color: #f59e0b; font-weight: bold; text-decoration: none;'>Eu</a>"
    eu_links.append(eu_html)
    
    # URL de NASA Exoplanet Archive: Usan el nombre codificado (%20 para espacios)
    nombre_nasa = urllib.parse.quote(nombre_str)
    url_nasa = f"https://exoplanetarchive.ipac.caltech.edu/overview/{nombre_nasa}"
    # Lo pintamos con su color corporativo azul
    nasa_html = f"<a href='{url_nasa}' target='_blank' style='color: #0b3d91; font-weight: bold; text-decoration: none;'>NASA</a>"
    nasa_links.append(nasa_html)

internal_data['eu_link'] = eu_links
internal_data['nasa_link'] = nasa_links

# Procesamos Status con colores (Píldoras visuales)
status_vals = []
for s in df.get('planet_status', pd.Series(['Confirmed']*len(df))):
    s_str = str(s).strip().title()
    if s_str == 'Confirmed':
        # Verde suave de fondo con texto verde oscuro
        status_vals.append("<span style='background-color: #d4edda; color: #155724; padding: 3px 8px; border-radius: 6px; font-weight: bold; font-size: 0.85em;'>Confirmed</span>")
    elif s_str == 'Candidate':
        # Naranja claro de fondo con texto naranja/marrón oscuro
        status_vals.append("<span style='background-color: #ffe8a1; color: #856404; padding: 3px 8px; border-radius: 6px; font-weight: bold; font-size: 0.85em;'>Candidate</span>")
    elif s_str in ['Retracted', 'False Positive', 'Discarded', 'Controversial']:
        # Rojo suave de alerta con texto granate oscuro (Para los descartados)
        status_vals.append(f"<span style='background-color: #f8d7da; color: #721c24; padding: 3px 8px; border-radius: 6px; font-weight: bold; font-size: 0.85em;'>{s_str}</span>")
    else:
        status_vals.append(s_str)
internal_data['status'] = status_vals

# Procesamos Coordenadas (RA y Dec)
ra_vals, dec_vals = [], []
for idx, row in df.iterrows():
    try:
        ra = float(row.get('ra', np.nan))
        ra_vals.append(f"{ra:.4f}" if pd.notna(ra) else "")
    except: ra_vals.append("")
    
    try:
        dec = float(row.get('dec', np.nan))
        dec_vals.append(f"{dec:.4f}" if pd.notna(dec) else "")
    except: dec_vals.append("")

internal_data['ra_val'] = ra_vals
internal_data['dec_val'] = dec_vals

# Procesamos Tipo Espectral
spt_vals, spt_refs = [], []
for idx, row in df.iterrows():
    # 1. Guardamos el valor del tipo espectral (ej. M4.5 V)
    spt_vals.append(str(row.get('star_sp_type', '')) if pd.notna(row.get('star_sp_type')) else "")  
    # 2. Cogemos la referencia cruda
    raw_ref = str(row.get('star_sp_type_ref', '')) if pd.notna(row.get('star_sp_type_ref')) else ""
    # 3. Le aplicamos la magia de los colores y la guardamos
    spt_refs.append(format_reference_smart(raw_ref))
internal_data['spt_val'] = spt_vals
internal_data['spt_ref'] = spt_refs

# Procesamos Columnas dinámicas (física y referencias)
for name, info in params.items():
    vals_col, errs_col, refs_col = [], [], []
    base_col_name = info['val']
    ref_col_name = f"{base_col_name}_ref" 
    
    for idx, row in df.iterrows():
        val = row.get(info['val'], np.nan)
        e_min = row.get(info['min'], np.nan)
        e_max = row.get(info['max'], np.nan)
        ref_str = row.get(ref_col_name, "")
        
        is_min_mass = False
        
        # SI ESTAMOS EN LA MASA DEL PLANETA Y ESTÁ VACÍA (O BORRADA CON NaN)
        if name == 'M_p' and pd.isna(val):
            # Leemos estricta y únicamente las columnas de mass_sini
            val = row.get('mass_sini', np.nan)
            e_min = row.get('mass_sini_error_min', np.nan)
            e_max = row.get('mass_sini_error_max', np.nan)
            ref_str = row.get('mass_sini_ref', "")
            
            # --- EL RESCATE DE LA REFERENCIA FANTASMA ---
            # Exoplanet.eu a veces pone el valor en mass_sini pero tira la referencia en mass_ref.
            ref_str = row.get('mass_sini_ref', "")
            if pd.isna(ref_str) or str(ref_str).strip() == "":
                ref_str = row.get('mass_ref', "")
            
            if pd.notna(val): is_min_mass = True
            
        v_str, e_str = format_val_and_error(val, e_min, e_max)
        
        # Añadimos la etiqueta visual
        if is_min_mass and v_str != "":
            v_str += ' <span style="color: #999; font-size: 0.8em; margin-left: 2px;">sin <i>i</i></span>'

        # Excepciones para referencias especiales
        if name == 'T_0,p': 
            ref_str = row.get('T0_val_ref', '') 
        elif name == '&#961;_star': 
                if pd.isna(ref_str) or str(ref_str).strip() == '':
                    ref_str = "-" 
            
        if pd.isna(ref_str): ref_str = ""

        # --- NUEVA REGLA DE LIMPIEZA ---
        # Si el valor final está vacío, aniquilamos la referencia
        if v_str.strip() == "":
            ref_str = ""
        
        vals_col.append(v_str)
        errs_col.append(e_str)
        refs_col.append(format_reference_smart(ref_str))
    
    internal_data[f'{name}_val'] = vals_col
    internal_data[f'{name}_err'] = errs_col
    internal_data[f'{name}_ref'] = refs_col

# -------------------------------------------------------------------------
# 3. CREACIÓN DEL DATAFRAME MULTI-ÍNDICE PARA HTML
# -------------------------------------------------------------------------
final_columns = []
final_data_arrays = []

header_id = f"ID<br><span style='font-size:0.8em; color:#666'>Planet</span>"
header_disc = f"Discovery<br><span style='font-size:0.8em; color:#666'>Ref</span>"
header_met = f"Detection<br><span style='font-size:0.8em; color:#666'>Method</span>"
header_status = f"Status<br><span style='font-size:0.8em; color:#666'>Planet</span>"


final_columns.append( (header_id, "") ); final_data_arrays.append(internal_data['id_planeta'])
final_columns.append( (header_disc, "") ); final_data_arrays.append(internal_data['discovery_ref'])
final_columns.append( (header_met, "") ); final_data_arrays.append(internal_data['metodo'])
final_columns.append( (header_status, "") ); final_data_arrays.append(internal_data['status'])


for name, info in params.items():
    # ¡AQUÍ INYECTAMOS RA Y DEC JUSTO ANTES DEL RADIO ESTELAR!
    if name == 'R_star':
        header_ra = f"RA<br><span style='font-size:0.85em; color:#666; font-weight:normal'>(deg)</span>"
        header_dec = f"Dec<br><span style='font-size:0.85em; color:#666; font-weight:normal'>(deg)</span>"
        final_columns.append((header_ra, "")); final_data_arrays.append(internal_data['ra_val'])
        final_columns.append((header_dec, "")); final_data_arrays.append(internal_data['dec_val'])

    unit_html = f"<span style='font-size:0.85em; color:#666; font-weight:normal'>({info['unit']})</span>" if info['unit'] else ""
    header_html = f"{name}<br>{unit_html}"
    
    final_columns.append((header_html, 'Value')); final_data_arrays.append(internal_data[f'{name}_val'])
    final_columns.append((header_html, 'Error')); final_data_arrays.append(internal_data[f'{name}_err'])
    final_columns.append((header_html, 'Ref'));   final_data_arrays.append(internal_data[f'{name}_ref'])

header_spt = "Sp. Type" # Puedes cambiar "SpT" por "Sp. Type" o lo que prefieras

# Y ahora sí, añadimos las columnas
final_columns.append( (header_spt, "Value") ); final_data_arrays.append(internal_data['spt_val'])
final_columns.append( (header_spt, "Ref") );   final_data_arrays.append(internal_data['spt_ref'])

# Añadimos las dos columnas de Bases de Datos al final de la tabla
header_db = "Databases<br><span style='font-size:0.8em; color:#666'>Links</span>"

final_columns.append( (header_db, "EU") );   final_data_arrays.append(internal_data['eu_link'])
final_columns.append( (header_db, "NASA") ); final_data_arrays.append(internal_data['nasa_link'])

df_final = pd.DataFrame(final_data_arrays).T
df_final.columns = pd.MultiIndex.from_tuples(final_columns)

# -------------------------------------------------------------------------
# 4. EXPORTAR A HTML INTERACTIVO
# -------------------------------------------------------------------------
print("Generando HTML...")
html_table = df_final.style \
    .set_uuid("myTable") \
    .set_table_attributes('class="display compact cell-border hover" style="width:100%"') \
    .format(na_rep="") \
    .hide(axis='index') \
    .to_html(escape=False)

full_html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>{table_title}</title>
    <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 20px; background-color: #f4f4f4; }}
        h2 {{ text-align: center; color: #333; margin-bottom: 20px; font-weight: 600; }}
        .table-container {{ width: 95%; margin: 0 auto; background-color: white; padding: 20px; border-radius: 8px; box-shadow: 0 0 15px rgba(0,0,0,0.1); }}
        .top-scrollbar-wrapper {{ width: 100%; overflow-x: auto; overflow-y: hidden; margin-bottom: 5px; }}
        .top-scrollbar-content {{ height: 1px; }}
        table.dataTable thead th {{ text-align: center; background-color: #2c3e50; color: white; font-size: 0.9em; vertical-align: bottom; border-bottom: 2px solid #1a252f; }}
        table.dataTable tbody td {{ font-size: 0.9em; text-align: center; vertical-align: middle; }}
        a {{ text-decoration: none; color: #007bff; transition: 0.2s; }}
        a:hover {{ text-decoration: underline; color: #0056b3; }}
    </style>
</head>
<body>
    <h2>{table_title}</h2>
    <div class="table-container">
        <div id="topScroll" class="top-scrollbar-wrapper">
            <div id="topScrollContent" class="top-scrollbar-content"></div>
        </div>
        {html_table}
    </div>
    <script src="https://code.jquery.com/jquery-3.7.0.min.js"></script>
    <script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
    <script>
        $(document).ready(function() {{
            var table = $('#T_myTable').DataTable({{
                "paging": true, "pageLength": 20, "lengthMenu": [[10, 20, 50, -1], [10, 20, 50, "All"]],
                "ordering": true, "info": true, "searching": true, "scrollX": true,
                "columnDefs": [ {{ "type": "html-num", "targets": "_all" }} ]
            }});
            var topScroll = $('#topScroll'); var topContent = $('#topScrollContent'); var bottomScroll = $('.dataTables_scrollBody'); 
            function updateScrollWidth() {{ topContent.width($('#T_myTable').width()); }}
            topScroll.on('scroll', function() {{ bottomScroll.scrollLeft($(this).scrollLeft()); }});
            bottomScroll.on('scroll', function() {{ topScroll.scrollLeft($(this).scrollLeft()); }});
            updateScrollWidth(); $(window).resize(updateScrollWidth); table.on('draw', updateScrollWidth);
        }});
    </script>
</body>
</html>
"""

output_filename = 'Tabla_Publica_Interactiva.html'
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(full_html)

print(f"¡Éxito! Abre '{output_filename}' en tu navegador.")

Leyendo catálogo maestro: Exoplanets_SolarNeighbourhood _Catalogue.csv...
Generando HTML...
¡Éxito! Abre 'Tabla_Publica_Interactiva.html' en tu navegador.
